# STRATA na GPU: Llama-3.1-8B no llama.cpp com CUDA, contra Q5_K_M e Q6_K (com e sem imatrix)

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta** (de preferência) ou **L4**. Depois *Executar tudo*.

O notebook:
1. compila o llama.cpp (commit fixo) com CUDA e com os tipos novos do STRATA (`kernel/ggml` e `kernel/cuda`): os níveis 1 e 2 do modo residual em blocos montados na carga;
2. confere o CUDA contra a CPU com um teste próprio (`tests/kernel/strata_cuda_check.c`): multiplicações de matriz em todos os tamanhos de lote, dos dois caminhos que a GPU usa (produto escalar na geração, desquantização + cuBLAS na leitura do prompt);
3. gera os GGUF do Llama-3.1-8B: BF16, Q4_K_M, Q5_K_M e Q6_K, cada um **sem** e **com** imatrix (calculada no wikitext-2 de *treino*, nunca no de teste), e os níveis 1 e 2 do STRATA em cima dos dois Q4_K_M;
4. confere o caminho de geração no modelo inteiro (perplexidade com lote de 8 tokens, que passa pelo produto escalar, contra lote normal);
5. mede a perplexidade no wikitext-2 de teste e a divergência KL contra o BF16, tudo na GPU;
6. mede tokens por segundo na GPU com o `llama-bench`, alternando os modelos em rodadas.

Ocupa ~110 GB de disco no pico (pesos originais, BF16 e 10 GGUF). A compilação com CUDA é a parte mais lenta do começo. Para medir só a velocidade, ponha `MEDIR_QUALIDADE = False` abaixo; para pular os modelos com imatrix, `COM_IMATRIX = False`.

In [ ]:
MEDIR_QUALIDADE = True   # perplexidade no wikitext inteiro + divergência KL; False mede só a velocidade
COM_IMATRIX = True       # também os k-quants e o STRATA feitos sobre um Q4_K_M com imatrix
CHUNKS = 0               # trechos de 512 tokens na perplexidade (0 = o texto inteiro)
KLD_CHUNKS = 100         # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125     # trechos de 512 tokens do wikitext de treino para a imatrix
RODADAS = 3              # rodadas alternadas do llama-bench
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 110:
    print('Aviso: pouco disco para todos os modelos. Ponha COM_IMATRIX = False ou KLD_CHUNKS menor.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-bench llama-perplexity llama-imatrix 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(bench|perplexity|quantize|imatrix)$|libggml-cuda'

Conferência do CUDA contra a CPU: multiplica matrizes de cada tipo (Q5_K e Q6_K como controle, e os 4 do STRATA) por lotes de 1, 2, 5, 8, 9, 64 e 512 colunas, nos dois backends, e compara os dois com uma referência em float64. Lotes de até 8 usam o produto escalar da geração; acima disso, a desquantização + cuBLAS. Tudo tem de sair OK.

In [ ]:
LIB = '/content/llama.cpp/build/bin'
!cc -O2 strata/tests/kernel/strata_cuda_check.c -I llama.cpp/ggml/include -L {LIB} -lggml -lggml-base -lggml-cpu -lggml-cuda -lm -Wl,-rpath,{LIB} -o strata_cuda_check
out = !./strata_cuda_check 2>&1
print('\n'.join(out))
assert any(l.strip() == 'tudo OK' for l in out), 'o CUDA deu resultado diferente da CPU: avise na thread (cole a saída acima)'

Os GGUF. Pesos originais → BF16 → k-quants com o `llama-quantize` padrão; os níveis do STRATA saem do Q4_K_M + pesos BF16 (na GPU, alguns minutos).

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
if COM_IMATRIX:
    !llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
    !tail -2 {G}/imatrix.log
SUFIXOS = [''] + (['_imat'] if COM_IMATRIX else [])
for s in SUFIXOS:
    extra = f'--imatrix {G}/imatrix.gguf' if s else ''
    for q in ['Q4_K_M', 'Q5_K_M', 'Q6_K']:
        !llama.cpp/build/bin/llama-quantize {extra} {G}/BF16.gguf {G}/{q}{s}.gguf {q} > {G}/{q}{s}.log 2>&1
%cd /content/strata
for s in SUFIXOS:
    !python tests/kernel/strata_gguf_niveis.py --model {src} --gguf {G}/Q4_K_M{s}.gguf --nivel 1 2 --out {G}/STRATA_N1{s}.gguf {G}/STRATA_N2{s}.gguf 2>&1 | grep -v Warning | tail -2
%cd /content
!rm -rf {src}
!ls -la {G}/*.gguf
!df -h /content | tail -1

In [ ]:
def ppl_de(m, chunks=0, extra=''):
    c = f'--chunks {chunks}' if chunks else ''
    out = !llama.cpp/build/bin/llama-perplexity -m {G}/{m}.gguf -f /content/wiki.test.raw -c 512 -ngl 99 {c} {extra} 2>&1
    r = re.search(r'Final estimate: PPL = ([0-9.]+)\s*\+/- ([0-9.]+)', '\n'.join(out))
    if not r:
        print(f'Erro ao obter a PPL de {m}. Fim da saída:')
        print('\n'.join(out[-10:]))
        return None
    return (float(r.group(1)), float(r.group(2)))

Conferência do caminho de geração no modelo inteiro: a perplexidade em 4 trechos com lote de 8 tokens (passa pelo produto escalar, o mesmo da geração) tem de dar quase o mesmo número que com lote normal (desquantização + cuBLAS). Os dois caminhos arredondam diferente, então uma diferença de até ~1% é normal (o Q5_K_M mostra o tamanho desse ruído); um erro de verdade daria dezenas de por cento.

In [ ]:
for m in ['Q5_K_M', 'STRATA_N1', 'STRATA_N2']:
    a, b = ppl_de(m, 4, '-b 8 -ub 8'), ppl_de(m, 4)
    print(f'{m}: lote 8 {a[0]:.4f}, lote normal {b[0]:.4f}, diferença {100 * (b[0] / a[0] - 1):+.3f}%')
    assert abs(b[0] / a[0] - 1) < 3e-2, 'os dois caminhos da GPU deram resultados diferentes: avise na thread'

Perplexidade no wikitext-2 de teste (contexto 512) e divergência KL contra o BF16 nos primeiros `KLD_CHUNKS` trechos. A KL mede quanto a distribuição do próximo token se afasta da do modelo original; é bem menos ruidosa que a perplexidade.

In [ ]:
GRUPO = ['Q4_K_M', 'Q5_K_M', 'STRATA_N1', 'Q6_K', 'STRATA_N2']
MODELOS = ['BF16'] + [m + s for s in SUFIXOS for m in GRUPO]
ppl, kld = {}, {}
if MEDIR_QUALIDADE:
    for m in MODELOS:
        ppl[m] = ppl_de(m, CHUNKS)
        if ppl[m]:
            print(m, ppl[m], f'ΔPPL {100 * (ppl[m][0] / ppl["BF16"][0] - 1):+.2f}%')
    !llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
    for m in MODELOS[1:]:
        out = !llama.cpp/build/bin/llama-perplexity -m {G}/{m}.gguf --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
        txt = '\n'.join(out)
        k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
        top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
        if not k_:
            print(f'Erro na KL de {m}:', '\n'.join(out[-8:]))
            continue
        kld[m] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
        print(m, 'KL ×1000', round(1000 * kld[m][0], 2), 'mesmo 1º token', kld[m][2], '%')
    !rm -f /content/kld_base.bin

Tokens por segundo na GPU com o `llama-bench` (todas as camadas na GPU): geração de 128 tokens (`tg128`) e leitura de um prompt de 512 tokens (`pp512`). Só os modelos sem imatrix (a imatrix muda os números dentro dos blocos, não a velocidade). Os modelos são alternados em ordem aleatória a cada rodada.

In [ ]:
import random, statistics
BENCH = 'llama.cpp/build/bin/llama-bench'
vel = {}
def bench(m, p, n):
    out = subprocess.run([BENCH, '-m', f'{G}/{m}.gguf', '-ngl', '99', '-p', str(p), '-n', str(n), '-r', '3', '-o', 'json'],
                         capture_output=True, text=True).stdout
    return json.loads(out)[0]['avg_ts']
for r in range(RODADAS):
    ordem = GRUPO[:]
    random.shuffle(ordem)
    for m in ordem:
        vel.setdefault((m, 'tg128'), []).append(bench(m, 0, 128))
        vel.setdefault((m, 'pp512'), []).append(bench(m, 512, 0))
    print(f'rodada {r + 1} de {RODADAS} pronta')
TESTES = ['tg128', 'pp512']
med = {k: statistics.median(v) for k, v in vel.items()}

In [ ]:
from IPython.display import Markdown, display
tam = {m: os.path.getsize(f'{G}/{m}.gguf') / 2**30 for m in MODELOS}
L = [f'# STRATA na GPU (Llama-3.1-8B, llama.cpp CUDA)', '',
     f'GPU: {GPU}. ' +
     (f'{TEXTO} de teste, contexto 512{f", {CHUNKS} trechos" if CHUNKS else ", inteiro"}; KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos; imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. ' if MEDIR_QUALIDADE else 'Só velocidade (qualidade não medida nesta rodada). ') +
     f'tok/s = mediana de {RODADAS} rodadas alternadas.', '',
     '| modelo | GGUF (GiB) | PPL | ΔPPL | KL (×1000) | mesmo 1º token | ' + ' | '.join(f'{t} (tok/s)' for t in TESTES) + ' |',
     '|---|---:|---:|---:|---:|---:|' + '---:|' * len(TESTES)]
for m in MODELOS:
    p, e = ppl.get(m) or (float('nan'), float('nan'))
    v = ' | '.join(f'{med[(m, t)]:.1f}' if (m, t) in med else '' for t in TESTES)
    kl = f'{1000 * kld[m][0]:.2f} ± {1000 * kld[m][1]:.2f} | {kld[m][2]:.2f}%' if m in kld else ' | '
    dp = f'{100 * (p / ppl["BF16"][0] - 1):+.2f}%' if ppl.get('BF16') else ''
    L.append(f'| {m} | {tam[m]:.3f} | {p:.4f} ± {e:.4f} | {dp} | {kl} | {v} |')
L += ['', 'STRATA contra o k-quant de mesma qualidade (velocidade: 1,00 = igual, abaixo de 1 = mais lento; KL: abaixo de 1 = mais perto do original):', '',
      '| comparação | tamanho | KL | ' + ' | '.join(TESTES) + ' |', '|---|---:|---:|' + '---:|' * len(TESTES)]
for s in SUFIXOS:
    for a, b in [('STRATA_N1', 'Q5_K_M'), ('STRATA_N2', 'Q6_K')]:
        kl = f'{kld[a + s][0] / kld[b + s][0]:.2f}' if a + s in kld and b + s in kld else ''
        v = ' | '.join(f'{med[(a, t)] / med[(b, t)]:.2f}' if not s else '' for t in TESTES)
        L.append(f'| {a}{s} / {b}{s} | {tam[a + s] / tam[b + s]:.3f} | {kl} | {v} |')
L += ['', 'Llama-3.2-1B na CPU (seção 23 dos resultados): nível 1 KL 14,1 vs Q5_K_M 13,8; nível 2 KL 5,1 vs Q6_K 7,0 (×1000).']
os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_gpu.md', 'w').write('\n'.join(L) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'chunks': CHUNKS, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'ppl': ppl, 'kld': kld,
           'tok_s': {f'{m}|{t}': v for (m, t), v in vel.items()}, 'gguf_gib': tam}, open('/content/res/strata_8b_gpu.json', 'w'), indent=1)
display(Markdown('\n'.join(L)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_gpu.md')
files.download('/content/res/strata_8b_gpu.json')